In [6]:
# Install CUDA C++ plugin for Colab:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [7]:
# Detect selected GPU and its NVIDA architecture:
import subprocess
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits")
if "not found" in gpu_info.lower(): raise RuntimeError("Error: No GPU found. Please select a GPU runtime environment.")
gpu_name, compute_cap = map(str.strip, gpu_info.split(','))
gpu_arch = f"sm_{compute_cap.replace('.', '')}"

print(f"{'GPU Name':<15}: {gpu_name}")
print(f"{'Architecture':<15}: {gpu_arch}")

GPU Name       : Tesla T4
Architecture   : sm_75


In [10]:
%%cuda -c "--gpu-architecture $gpu_arch"
#include <stdio.h>
#include <iostream>
#include <cstdlib> // For rand() and srand()
#include <ctime>   // For time()

using namespace std;

__global__ void MatrixMult(int* d_a, int* d_b, int* d_c, int n) {
   int row = blockIdx.y * blockDim.y + threadIdx.y;
   int col = blockIdx.x * blockDim.x + threadIdx.x;
   int temp = 0;
   if (row < n && col < n) {
       for (int i = 0; i < n; i++) {
           temp += d_a[row * n + i] * d_b[i * n + col]; // 1D indexing
       }
       d_c[row * n + col] = temp; // 1D indexing
   }
}

void fillarray(int* a, int n) {
    for (int k = 0; k < n; k++) {
        for (int i = 0; i < n; i++) {
            a[k * n + i] = (std::rand() % 10 + 1);
        }
    }
}

void printHostMatrix(int* h_matrix, int n) {
    for (int r = 0; r < n; ++r) {
        for (int c = 0; c < n; ++c) {
            std::cout << h_matrix[r * n + c] << "\t"; // Use tab for better spacing
        }
        std::cout << std::endl;
    }
    std::cout << std::endl;
}

int main() {
    // Seed the random generator ONCE here to prevent Matrix A and B from being identical
    std::srand(std::time(0));

    int N = 3;
    dim3 blockSize(N, N); // N threads per block in X and Y
    dim3 gridSize(1, 1);

    int* h_a = new int[N * N];
    int* h_b = new int[N * N];
    int* h_c = new int[N * N];

    int* d_a, *d_b, *d_c;
    int size = sizeof(int) * (N * N);

    // fill the arrays
    fillarray(h_a, N);
    fillarray(h_b, N);

    // Allocate device memory
    cudaMalloc((void**)&d_a, size);
    cudaMalloc((void**)&d_b, size);
    cudaMalloc((void**)&d_c, size);

    // Copy host arrays to device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Launch matrix multiplication kernel
    MatrixMult<<<gridSize, blockSize>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize(); // Wait for kernel to finish

    // Copy result back from device to host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Print input matrices and result matrix
    std::cout << "Matrix A:" << std::endl;
    printHostMatrix(h_a, N);
    std::cout << "Matrix B:" << std::endl;
    printHostMatrix(h_b, N);
    std::cout << "Matrix C (Result A * B):" << std::endl;
    printHostMatrix(h_c, N);

    // Free device memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    // Free host memory
    delete[] h_a;
    delete[] h_b;
    delete[] h_c;

    return 0;
}

Matrix A:
1	6	3	
4	4	4	
1	10	3	

Matrix B:
5	10	2	
8	10	2	
9	2	9	

Matrix C (Result A * B):
80	76	41	
88	88	52	
112	116	49	


